# Kingfisher - EA-LSTM training on a GPU (P5.1)

Trains 2 targets x 5 seeds = 10 EA-LSTM members (NeuralHydrology 1.13). About 1-1.5 h on a T4.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU -> Save`.

Then run the cells top to bottom (Shift+Enter). Each cell says what it does and what you should see.

In [ ]:
# STEP 1 - check the GPU. You should see a line starting with 'GPU 0: Tesla T4'.
# If you see an error instead: Runtime -> Change runtime type -> T4 GPU, then run this again.
!nvidia-smi -L

In [ ]:
# STEP 2 - connect Google Drive (click 'Connect' / 'Allow' in the pop-up).
# Trained models are saved to Drive AS THEY FINISH, so a Colab disconnect loses nothing:
# just re-run all cells and training continues where it stopped.
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE = '/content/drive/MyDrive/kingfisher_ealstm'
os.makedirs(SAVE, exist_ok=True)
print('results will be kept in', SAVE)

In [ ]:
# STEP 3 - upload kingfisher_colab.zip (made on your PC by scripts/make_colab_bundle.py).
# Click 'Choose Files' below and pick dist/kingfisher_colab.zip. Wait for 100%.
# (Faster alternative: put the zip in your Google Drive's MyDrive folder first - this
#  cell will find it there and skip the upload.)
import os, shutil
ZIP = '/content/kingfisher_colab.zip'
if os.path.exists('/content/drive/MyDrive/kingfisher_colab.zip'):
    shutil.copy('/content/drive/MyDrive/kingfisher_colab.zip', ZIP)
    print('found the zip in Drive')
elif not os.path.exists(ZIP):
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    if name != 'kingfisher_colab.zip':
        os.rename(name, ZIP)
print('zip size (MB):', round(os.path.getsize(ZIP) / 1e6))

In [ ]:
# STEP 4 - unpack and install (about 1-2 minutes). Ends with 'ready'.
!rm -rf /content/kingfisher && mkdir -p /content/kingfisher
!unzip -q /content/kingfisher_colab.zip -d /content/kingfisher
!pip -q install neuralhydrology==1.13.0 pydantic-settings structlog netCDF4 pyyaml
%cd /content/kingfisher
# keep the trained models in Drive (survives disconnects)
!mkdir -p artifacts && rm -rf artifacts/ealstm && ln -s /content/drive/MyDrive/kingfisher_ealstm artifacts/ealstm
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
print('ready')

In [ ]:
# STEP 5 - 2-minute check: 1 epoch on 4 reaches on the GPU. Must end with a JSON block
# containing "p10_p50_p90". If it shows a red error instead, STOP and send the error.
!python -u -m models.ealstm smoke --device cuda:0 2>&1 | tail -25

In [ ]:
# STEP 6 - the real training: turbidity x 5 seeds, then NDCI x 5 seeds (~1-1.5 h).
# Keep this tab open. If Colab disconnects: Runtime -> Run all. Members already
# finished are skipped (--skip-trained), so it resumes.
# You will see a 'train_start' line per member, epoch losses, then 'train_done' (~5-10 min each).
!python -u -m models.ealstm train --target all --device cuda:0 --skip-trained 2>&1 | grep --line-buffered -E 'train_start|train_done|skip_trained|Error|error|Traceback|average loss'

In [ ]:
# STEP 7 - check: all 10 members must show an epoch number, none 'NOT TRAINED'.
!python -m models.ealstm status

In [ ]:
# STEP 8 - zip the trained models and download ealstm_runs.zip to your PC.
# (It is also safe in Google Drive -> MyDrive/kingfisher_ealstm.)
!cd /content/kingfisher && rm -f /content/ealstm_runs.zip && zip -qr /content/ealstm_runs.zip artifacts/ealstm -x 'artifacts/ealstm/smoke/*' '*optimizer_state_*'
!ls -la /content/ealstm_runs.zip
from google.colab import files
files.download('/content/ealstm_runs.zip')